# Aggregations

In [154]:
# going from clean 2024 data 
import os 
os.getcwd()
os.chdir("/Users/ellawileman/Documents/last_sem/dsci_capstone/final_repo/US-Census-Voting-and-Registration-DSCI-Capstone-Project/clean_data")

In [155]:
import pandas as pd
data = pd.read_csv("nov24pub_clean.csv") # replace with your own path to clean data on your machine

In [156]:
data["PEHSPNON"]

0        2
1        2
2        2
3        2
4        2
        ..
62404    2
62405    2
62406    2
62407    2
62408    2
Name: PEHSPNON, Length: 62409, dtype: int64

In [157]:
data.head()

,HRHHID,HRMONTH,HRYEAR4,HURESPLI,HUFINAL,FILLER,HETENURE,HEHOUSUT,HETELHHD,HETELAVL,...,PRSUPINT,PES1,PES2,PES3,PES4,PES5,PES6,PES7,PRS8,PESCK4
0,920760069071210,11,2024,2,201,NaN,1,1,1,-1,...,1,1,1,-1,-1,1,1,6,4,2
1,920760069071210,11,2024,2,201,NaN,1,1,1,-1,...,1,1,1,-1,-1,1,1,6,4,1
2,50057098511186,11,2024,1,201,NaN,1,1,1,-1,...,1,1,1,-1,-1,1,1,9,4,1
3,6110057715727,11,2024,2,201,NaN,1,1,1,-1,...,1,1,1,-1,-1,1,1,-2,4,1
4,6110057715727,11,2024,2,201,NaN,1,1,1,-1,...,1,1,1,-1,-1,1,1,-2,4,1


In [158]:
# identifier for state
data["GESTFIPS"].value_counts()

GESTFIPS
6     4953
48    3278
12    2564
36    2071
42    1918
17    1645
54    1556
39    1538
26    1447
37    1365
25    1308
16    1291
22    1280
30    1243
47    1227
51    1205
13    1203
1     1196
53    1175
34    1172
28    1164
5     1138
45    1068
41    1063
35    1054
40    1036
15    1035
49    1031
56    1022
55    1020
18    1019
11    1014
29    1007
38     996
50     964
27     921
33     861
19     845
4      840
20     833
32     831
31     817
46     754
21     737
24     733
8      728
10     726
9      710
2      680
23     569
44     558
Name: count, dtype: int64

In [159]:
data = data.rename(columns={
    "PRTAGE": "age",
    "PESEX": "sex",
    "PEMARITL": "marital_status",
    "PEEDUCA": "education",
    "HEFAMINC": "family_income",
    "PRCHLD": "number_of_children",
    'PTDTRACE': 'race',
    'PEHSPNON': 'hispanic_flag',
    'PEAFEVER': 'veteran',
    'PWSSWGT': 'weight',
    'GESTFIPS': 'state'

})

In [160]:
# subset only a few features for now, can add more in later
data_subset = data[["age","sex", "marital_status","education","family_income","number_of_children","race","hispanic_flag", "veteran",'state',"weight"]]
data_subset

,age,sex,marital_status,education,family_income,number_of_children,race,hispanic_flag,veteran,state,weight
0,73,2,1,39,10,0,1,2,2,1,18354163
1,76,1,1,43,10,0,1,2,1,1,15592437
2,85,2,3,39,15,0,1,2,2,1,16356249
3,67,1,1,40,16,0,1,2,2,1,20773863
4,66,2,1,40,16,0,1,2,2,1,16796510
...,...,...,...,...,...,...,...,...,...,...,...
62404,69,1,1,40,16,0,1,2,2,54,7651462
62405,65,2,1,39,16,0,1,2,2,54,6827271
62406,66,1,1,41,12,0,1,2,2,56,3872403
62407,73,1,1,39,13,0,1,2,2,56,2790829


In [161]:
bins = [18, 25, 35, 45, 55, 65, 100]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]

data_subset["age_group"] = pd.cut(
    data_subset["age"],
    bins=bins,
    labels=labels,
    right=False
)

/var/folders/yz/btvl331d4g54624wdzgx_2hc0000gn/T/ipykernel_21423/2537900120.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_subset["age_group"] = pd.cut(


In [162]:
data_subset

,age,sex,marital_status,education,family_income,number_of_children,race,hispanic_flag,veteran,state,weight,age_group
0,73,2,1,39,10,0,1,2,2,1,18354163,65+
1,76,1,1,43,10,0,1,2,1,1,15592437,65+
2,85,2,3,39,15,0,1,2,2,1,16356249,65+
3,67,1,1,40,16,0,1,2,2,1,20773863,65+
4,66,2,1,40,16,0,1,2,2,1,16796510,65+
...,...,...,...,...,...,...,...,...,...,...,...,...
62404,69,1,1,40,16,0,1,2,2,54,7651462,65+
62405,65,2,1,39,16,0,1,2,2,54,6827271,65+
62406,66,1,1,41,12,0,1,2,2,56,3872403,65+
62407,73,1,1,39,13,0,1,2,2,56,2790829,65+


In [163]:
# encode the features we don't want to group by in the rows
data_subset = data_subset.drop(columns=["age"])
dummies_df = pd.get_dummies(data_subset, columns=["marital_status", "education","family_income", "number_of_children","veteran"])

# convert the dummies into binary variables for grouping
dummies_df = dummies_df.apply(lambda x: (x > 0).astype(int) if x.name.startswith(('marital_status', 'education', 'family_income', 'number_of_children', 'veteran')) else x)

# multiply the weight column by the dummy variables to get the weighted count for each group
for col in dummies_df.columns:
    if col.startswith(('marital_status', 'education', 'family_income', 'number_of_children', 'veteran')):
        dummies_df[col] = dummies_df[col] * dummies_df["weight"]

# for each group, the proportion of each category is the sum of the weighted counts for that category divided by the total weight for that group
grouped = dummies_df.groupby(["age_group", "sex", "race"]).sum()

/var/folders/yz/btvl331d4g54624wdzgx_2hc0000gn/T/ipykernel_21423/2089487947.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = dummies_df.groupby(["age_group", "sex", "race"]).sum()


In [164]:
grouped.head(50)

hispanic_flag   state        weight  marital_status_1  \
age_group sex race                                                          
18-24     1   1              4002   65613   80372636809        3872165356   
              2               580    8310   14489127983          69422956   
              3                75    1082    1763439136         208965923   
              4               282    3727    5064508524         147015017   
              5                22     204     337806600                 0   
              6                65    1199    1950320541                 0   
              7                38     645     991909117                 0   
              8                35     452     933797208                 0   
              9                 4      30      22065372                 0   
              10                8     151     204437291          98601014   
              11                4      44     116944431                 0   
              12                0       0             0                 0   
              13                0       0             0                 0   
              14                0       0             0                 0   
              15                6      45      42040020                 0   
              16                0       0             0                 0   
              17                1      15      11636891                 0   
              18                1      15      16915965                 0   
              19                4      57     165867555                 0   
              20                0       0             0                 0   
              21                9      75      74585747                 0   
              22                0       0             0                 0   
              25                0       0             0                 0   
              26                2      15      19815342                 0   
          2   1              3927   65562   81344106704        6923348629   
              2               597    8217   16050045849         598857379   
              3                83    1397    1709370767          88582451   
              4               281    3359    5622075321         200910032   
              5                28     378     676219668                 0   
              6                68    1076    1856062071         135387304   
              7                50     753    1193848987          93902799   
              8                52     625    1526668863                 0   
              9                 2      55      59646116                 0   
              10                2      24      82068009                 0   
              11                6     121     226977842                 0   
              12                0       0             0                 0   
              13                0       0             0                 0   
              14                0       0             0                 0   
              15                2      15      13484180                 0   
              16                3      79      95706802                 0   
              17                0       0             0                 0   
              18                0       0             0                 0   
              19                0       0             0                 0   
              20                0       0             0                 0   
              21               10      72     101942287           7973263   
              22                0       0             0                 0   
              25                0       0             0                 0   
              26                2      15      17435763                 0   
25-34     1   1              6385  102180  120817176653       45958301433   
              2               837   11745   22046754556        4678115103   

               

In [165]:
grouped_nozeros = grouped[grouped["weight"] > 0]
grouped_nozeros

hispanic_flag  state       weight  marital_status_1  \
age_group sex race                                                        
18-24     1   1              4002  65613  80372636809        3872165356   
              2               580   8310  14489127983          69422956   
              3                75   1082   1763439136         208965923   
              4               282   3727   5064508524         147015017   
              5                22    204    337806600                 0   
...                           ...    ...          ...               ...   
65+       2   10               15    208    244367834         101323456   
              13                4     37     24041704                 0   
              15                8     60     35975924          21954224   
              16               10    158    131158944          38764318   
              21               16    117    113188985         104096077   

                    marital_status_2  marital_status_3  marital_status_4  \
age_group sex race                                                         
18-24     1   1            452350124          25493792         212244545   
              2             63508286          69083703                 0   
              3             32797217                 0           8072252   
              4                    0                 0                 0   
              5                    0                 0                 0   
...                              ...               ...               ...   
65+       2   10                   0          34083740          42680279   
              13                   0                 0          24041704   
              15                   0           7010850           7010850   
              16                   0          69289394          12114653   
              21                   0           9092908                 0   

                    marital_status_5  marital_status_6  education_31  ...  \
age_group sex race                                                    ...   
18-24     1   1            433153816       75377229176      39494282  ...   
              2            189805379       14097307659             0  ...   
              3                    0        1513603744             0  ...   
              4             43720166        4873773341             0  ...   
              5                    0         337806600             0  ...   
...                              ...               ...           ...  ...   
65+       2   10            56131796          10148563             0  ...   
              13                   0                 0             0  ...   
              15                   0                 0             0  ...   
              16                   0          10990579             0  ...   
              21                   0                 0             0  ...   

                    number_of_children_8  number_of_children_9  \
age_group sex race                                               
18-24     1   1                 13719993                     0   
              2                        0                     0   
              3                        0                     0   
              4                        0                     0   
              5                        0                     0   
...                                  ...                   ...   
65+       2   10                       0                     0   
              13                       0                     0   
              15                       0                     0   
              16                       0                     0   
              21                       0                     0   

                    number_of_children_10  number_of_children_11  \
age_group sex race                                                 
18-24     1   1                         0               50

In [166]:
feature_cols = [c for c in grouped_nozeros.columns if c != "weight"]

grouped_nozeros[feature_cols] = grouped_nozeros[feature_cols].div(grouped_nozeros["weight"], axis=0)

/var/folders/yz/btvl331d4g54624wdzgx_2hc0000gn/T/ipykernel_21423/1089809800.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  grouped_nozeros[feature_cols] = grouped_nozeros[feature_cols].div(grouped_nozeros["weight"], axis=0)


In [167]:
grouped_nozeros.head(50)

hispanic_flag         state        weight  \
age_group sex race                                              
18-24     1   1      4.979307e-08  8.163599e-07   80372636809   
              2      4.003001e-08  5.735335e-07   14489127983   
              3      4.253053e-08  6.135738e-07    1763439136   
              4      5.568161e-08  7.359056e-07    5064508524   
              5      6.512602e-08  6.038958e-07     337806600   
              6      3.332785e-08  6.147707e-07    1950320541   
              7      3.830996e-08  6.502612e-07     991909117   
              8      3.748137e-08  4.840451e-07     933797208   
              9      1.812795e-07  1.359596e-06      22065372   
              10     3.913180e-08  7.386128e-07     204437291   
              11     3.420428e-08  3.762471e-07     116944431   
              15     1.427211e-07  1.070409e-06      42040020   
              17     8.593361e-08  1.289004e-06      11636891   
              18     5.911575e-08  8.867363e-07      16915965   
              19     2.411563e-08  3.436477e-07     165867555   
              21     1.206665e-07  1.005554e-06      74585747   
              26     1.009319e-07  7.569892e-07      19815342   
          2   1      4.827639e-08  8.059834e-07   81344106704   
              2      3.719616e-08  5.119612e-07   16050045849   
              3      4.855588e-08  8.172598e-07    1709370767   
              4      4.998154e-08  5.974662e-07    5622075321   
              5      4.140666e-08  5.589900e-07     676219668   
              6      3.663671e-08  5.797220e-07    1856062071   
              7      4.188134e-08  6.307330e-07    1193848987   
              8      3.406109e-08  4.093881e-07    1526668863   
              9      3.353110e-08  9.221053e-07      59646116   
              10     2.437003e-08  2.924404e-07      82068009   
              11     2.643430e-08  5.330917e-07     226977842   
              15     1.483220e-07  1.112415e-06      13484180   
              16     3.134573e-08  8.254377e-07      95706802   
              21     9.809472e-08  7.062820e-07     101942287   
              26     1.147068e-07  8.603008e-07      17435763   
25-34     1   1      5.284845e-08  8.457407e-07  120817176653   
              2      3.796477e-08  5.327315e-07   22046754556   
              3      4.550768e-08  7.178469e-07    2043611324   
              4      5.881114e-08  7.058534e-07    8348758767   
              5      4.371210e-08  4.126422e-07    1143848068   
              6      3.623200e-08  6.487792e-07    1766394572   
              7      4.053770e-08  6.675783e-07    1159414521   
              8      4.169193e-08  5.194120e-07    1151301808   
              9      3.065260e-08  8.122940e-07      65247311   
              10     2.878059e-08  5.612215e-07      69491276   
              11     2.877273e-08  2.249505e-07     382306396   
              13     2.576748e-08  7.730243e-08      77617227   
              14     1.155803e-08  5.547854e-07     173039894   
              15     1.312622e-07  9.844668e-07      45710024   
              16     1.915679e-08  5.842821e-07     208803234   
              17     2.017130e-07  1.311135e-06      19830150   
              20     1.346652e-07  1.009989e-06      14851643   
              21     1.454267e-07  1.090700e-06      55010538   

                    marital_status_1  marital_status_2  marital_status_3  \
age_group sex race                                                         
18-24     1   1             0.048178          0.005628          0.000317   
              2             0.004791          0.004383          0.004768   
              3             0.118499          0.018598          0.000000   
              4             0.029028          0.000000          0.000000   
              5             0.000000          0.000000          0.000000   
              6             0.000000          0.000000          0.000000   
              7             0.0

In [168]:
grouped_nozeros.shape

(195, 59)

In [169]:
# 200 rows x 20 years ~ 4000 rows 

# we dropped the zero weight rows so this may vary from year to year

In [170]:
# JUST DOING A SMALL FEATURE SET FOR NOW 
# CAN INCLUDE MORE LATER (AND INCLUDE A FORMAL FEATURE SELECTION)


# note: need to decrease cardinality of columns we use so it doesnt explode to so many demographic proportion columns

# doing this on 2024, can apply to all other years as seperate time snapshots. 



In [171]:
data_subset.nunique()

sex                       2
marital_status            6
education                16
family_income            16
number_of_children       16
race                     24
hispanic_flag             2
veteran                   2
state                    51
weight                39354
age_group                 6
dtype: int64

In [172]:
# reduce cardinality

In [173]:

data_subset["race"].value_counts()

race
1     51596
2      5687
4      2987
3       708
7       364
6       306
5       246
8       233
10       51
9        50
15       47
21       44
16       31
11       23
19        8
17        6
14        4
13        4
20        4
18        3
26        3
12        2
22        1
25        1
Name: count, dtype: int64

In [174]:
data_subset.columns

Index(['sex', 'marital_status', 'education', 'family_income',
       'number_of_children', 'race', 'hispanic_flag', 'veteran', 'state',
       'weight', 'age_group'],
      dtype='object')

In [175]:
data_subset['hispanic_flag'].mask(data_subset['hispanic_flag'] == 1, 'Hispanic', inplace=True)
data_subset['hispanic_flag'].mask(data_subset['hispanic_flag'] == 2, 'Non-Hispanic', inplace=True)

/var/folders/yz/btvl331d4g54624wdzgx_2hc0000gn/T/ipykernel_21423/130747232.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_subset['hispanic_flag'].mask(data_subset['hispanic_flag'] == 1, 'Hispanic', inplace=True)
/var/folders/yz/btvl331d4g54624wdzgx_2hc0000gn/T/ipykernel_21423/130747232.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Hispanic' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  d

In [176]:
# group most common races 
data_subset.loc[data_subset['race'] == 1, 'race_grouped'] = 'white_only'
data_subset.loc[data_subset['race'] == 2, 'race_grouped'] = 'black_only'
data_subset.loc[data_subset['race'] == 3, 'race_grouped'] = 'american_indian_only'
data_subset.loc[data_subset['race'] == 4, 'race_grouped'] = 'asian_only'
data_subset.loc[data_subset['race'] == 5, 'race_grouped'] = 'white_only'
data_subset.loc[(data_subset['race'] != 1) & (data_subset['race'] != 2) & (data_subset['race'] != 3) & (data_subset['race'] != 4) & (data_subset['race'] != 5), 'race_grouped'] = 'multiracial'

data_subset = data_subset.drop(columns="race")

In [177]:
data_subset["family_income"].value_counts()

family_income
16    14271
15    11077
14     8611
13     6281
12     4489
11     4030
10     2495
9      2435
7      1956
8      1874
6      1290
1       987
4       975
5       859
2       404
3       375
Name: count, dtype: int64

In [178]:
# Low class
# categories 1-6
# up to 20k
data_subset.loc[(data_subset['family_income'] == 1) | (data_subset['family_income'] == 2) | (data_subset['family_income'] == 3) | (data_subset['family_income'] == 4) | (data_subset['family_income'] == 5) | (data_subset['family_income'] == 6), 'family_income_grouped'] = 'low_income'

# Lower middle class
# categories 7-10
data_subset.loc[(data_subset['family_income'] == 7) | (data_subset['family_income'] == 8) | (data_subset['family_income'] == 9) | (data_subset['family_income'] == 10), 'family_income_grouped'] = 'lower_middle_class'


# True middle class
# categories 11-13
data_subset.loc[(data_subset['family_income'] == 11) | (data_subset['family_income'] == 12) | (data_subset['family_income'] == 13), 'family_income_grouped'] = 'middle_class'


# Upper middle class
# categories 14-15
data_subset.loc[(data_subset['family_income'] == 14) | (data_subset['family_income'] == 15), 'family_income_grouped'] = 'upper_middle_class'

# Upper class
# category 16
# note the cutoff is 150k in census data so we cannot differentiate further

data_subset.loc[data_subset['family_income'] == 16, 'family_income_grouped'] = 'high_income'

# drop original column
data_subset = data_subset.drop(columns="family_income")

In [179]:
data_subset["number_of_children"].value_counts()

number_of_children
0     48677
3      3689
4      2578
10     1814
1      1416
8      1223
2       868
5       803
6       478
11      356
14      184
9        93
13       75
7        66
15       54
12       35
Name: count, dtype: int64

In [180]:
# reduce cardinality of number of children 

# No children
data_subset.loc[data_subset['number_of_children'] == 0, 'children_grouped'] = 'No children'
# 1-2 children
data_subset.loc[(data_subset['number_of_children'] == 1) | (data_subset['number_of_children'] == 2), 'children_grouped'] = '1-2 children'

# 3+ children
data_subset.loc[data_subset['number_of_children'] > 2, 'children_grouped'] = '3+ children'

# drop original column and keep group
data_subset = data_subset.drop(columns="number_of_children")

In [181]:
# reduce cardinality of education feature

In [182]:
data_subset.nunique()

sex                          2
marital_status               6
education                   16
hispanic_flag                2
veteran                      2
state                       51
weight                   39354
age_group                    6
race_grouped                 5
family_income_grouped        5
children_grouped             3
dtype: int64